# Research Framework
### European Football Discipline Statistics — Scientific and Methodological Overview

## Purpose of This Notebook

This notebook does not contain analysis. Its purpose is to lay out the scientific goals of the project, describe the data available, and define the analytical approach — including its constraints.

The reason for writing this down explicitly is methodological: in data science, it is easy to let the analysis drift based on what the data happens to show. Writing down what we set out to do, and why, before touching the data makes the subsequent analysis more honest. It forces a clear distinction between what we planned to test and what we ended up being able to test.

## Scientific Goal

The central question of this project is: **is there a statistically significant difference in disciplinary outcomes for a given team compared to others?**

By *disciplinary outcomes* we mean primarily the number of yellow and red cards issued. We treat these as the observable result of a disciplinary decision — one that could, in principle, vary across teams for reasons beyond their underlying behavior on the pitch (number of fouls committed, style of play, competition level).

The analysis will look at this question from two angles:

1. **Within a competition** — does a team receive cards at a rate that is significantly different from other teams in the same league or tournament?
2. **Across competitions** — if a team appears as an outlier in a national league, does the same pattern appear in international competitions?

The second angle is a consistency check: if anomalous behavior is specific to one league, it may reflect the competitive context rather than something systematic. If it persists across competitions, the finding is more robust.

## The Data

Data was collected from two separate sources, as no single source covering both national and international competitions was found. This is already a constraint on the analysis, and it is worth being explicit about it upfront.

### National League Data
- Match-level records from multiple European national leagues
- Covers 10 seasons
- Each record corresponds to one match, with per-team values for fouls committed, yellow cards, and red cards

### International Tournament Data
- Season-aggregate records from European competitions (e.g. UEFA Champions League, Europa League)
- Each record is a season-level summary per team: total fouls, total yellow cards, total red cards for that season
- No match-level granularity available

### The Granularity Mismatch

These two datasets are not directly comparable as they stand. National data is at match level; international data is at season level. Running the same statistical test on both would be methodologically incorrect: a ratio computed from a single match (e.g. 2 fouls, 1 card) carries very different uncertainty than one computed from an entire season (e.g. 180 fouls, 42 cards).

The solution is to **aggregate the national data to season level** before any cross-competition comparison — summing fouls and cards per team per season. This puts both datasets on the same scale. The trade-off is that within-season variance is lost, but the comparison becomes valid.

Within-competition analysis (comparing teams within the same national league) will still use match-level data, where the granularity is an advantage.

## Analytical Approach

### Normalizing Cards by Fouls

A team that commits more fouls will, all else equal, receive more cards. To isolate the question of whether a team is *treated differently* — rather than simply *behaving more aggressively* — we normalize cards by fouls. The metric of interest becomes: **given the number of fouls committed, how many cards were issued?**

The simplest form is the ratio `cards / fouls`. This is intuitive and useful for initial exploration, but it has a statistical limitation: the ratio is more uncertain when computed from few fouls, and treating it as a continuous variable ignores that both numerator and denominator are counts.

A more rigorous approach — which we will use for formal testing — is to model cards as a count variable (Poisson or negative binomial distribution) with fouls entering as an **offset**. An offset tells the model to expect cards to scale proportionally with fouls, so that any deviation from that baseline is attributed to the team. This approach is standard in epidemiology and sports analytics for exactly this type of rate comparison.

For an initial exploration, a Kruskal-Wallis or Mann-Whitney test on the per-match ratio is a reasonable first step: it is non-parametric (no normality assumption required) and allows comparison of one team against the rest.

### Yellow and Red Cards Are Treated Separately

Yellow and red cards are not interchangeable. Red cards are rare events — in any given match, most teams receive zero — which means their distribution is very different from yellow cards and they require separate modeling. Aggregating them into a combined "cards" total would obscure this distinction.

## Known Limitations

### Selection Bias in the Cross-Competition Check

Not all teams appear in both datasets. Teams in European competitions are the strongest teams in their respective leagues — a non-random subset. This means we cannot directly generalize from national to international results: if a top club behaves differently in Europe than domestically, this could be explained by stronger opponents, higher stakes, or different refereeing standards, not only by the team's intrinsic behavior.

This limitation does not invalidate the cross-competition check, but it must be acknowledged when interpreting results.

### Other Possible Confounders

The cards/fouls ratio controls for foul frequency, but other factors that could influence disciplinary outcomes are not directly measurable from the available data: referee identity, home vs. away status, match importance, the score at the time of the foul, or the reputation of individual players. A finding of "team X receives more cards per foul" is consistent with several explanations, and the analysis should not claim more than the data supports.

## A Note on Method: Planning, Adapting, Concluding

This project is also intended as a demonstration of how data analysis actually works in practice — not as it is often presented in textbooks, but as it is done.

Three principles guide the work:

**1. Draw conclusions only from data you can trust.**  
Before any analysis, we verify the data: null values, duplicates, outliers, internal consistency. These checks are not a formality — they are a prerequisite. An outlier in the data could reflect a genuine extreme case, or it could reflect a scraping error. The analysis cannot proceed until we know which.

**2. With a goal set in advance, you may only partially achieve it.**  
We started with a research question. The data we were able to collect places real constraints on how fully we can answer it. Where those constraints are hit, we will say so explicitly, rather than silently working around them or overstating what the results show.

**3. Data analysis is a work in progress: start with a plan, but be ready to adapt.**  
There are things that cannot be planned in advance. A dataset that looks clean in theory may have structural problems in practice. An analysis that seemed straightforward may reveal something unexpected that changes the question. The plan matters — it keeps the work honest — but rigidity is a liability. What distinguishes good analytical practice is not following a fixed script, but knowing *why* you are deviating from it.

Throughout the analysis notebooks, these adaptation moments will be narrated explicitly: when the data forces a change of approach, that change will be documented, with the reason, so the reader can follow the reasoning rather than just the result.